
# Aula Prática — Chatbot com NLP e Aprendizado de Máquina

**Curso:** Ciência da Computação - Uniderp

**Tema:** Aprendizado de Máquina + NLU/NLG  

**Ambiente:** Jupyter Notebook / JupyterLab / Google Colab

**Professor:** Murilo Gustavo Nabarrete Costa

## Objetivo
Construir, passo a passo, um chatbot simples para atendimento de uma loja virtual.




## Arquitetura que vamos implementar

```text
Mensagem do usuário
        ↓
Pré-processamento
        ↓
Representação numérica do texto
        ↓
Classificador de intenção
        ↓
Intenção + entidade
        ↓
Gerador de resposta
        ↓
Resposta do chatbot
```

### Intenções

Nosso chatbot terá quatro intenções:

- `saudacao`
- `preco`
- `estoque`
- `pedido`

Exemplos:

| Mensagem | Intenção |
|---|---|
| "Oi" | saudacao |
| "Quanto custa o notebook?" | preco |
| "Tem o celular em estoque?" | estoque |
| "Quero saber onde está meu pedido" | pedido |

A ideia central é que **mensagens diferentes podem representar a mesma intenção**.


In [ ]:

import re
import random
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report



## Criando os dados de treinamento

Em **aprendizado supervisionado**, o modelo recebe exemplos de entrada associados a uma saída conhecida.

Aqui:

- entrada = mensagem do usuário;
- saída = intenção correta.

Esse conjunto pequeno será nosso **dataset de treinamento**.

> Em um sistema real, precisaríamos de muito mais exemplos e de uma estratégia cuidadosa para construção e validação do dataset.


In [ ]:

dados = [
    # saudacao
    ("oi", "saudacao"),
    ("olá", "saudacao"),
    ("bom dia", "saudacao"),
    ("boa tarde", "saudacao"),
    ("boa noite", "saudacao"),
    ("oi tudo bem", "saudacao"),
    ("olá gostaria de ajuda", "saudacao"),

    # preco
    ("qual o preço do notebook", "preco"),
    ("quanto custa o celular", "preco"),
    ("qual o valor desse produto", "preco"),
    ("quero saber o preço do computador", "preco"),
    ("me informe o valor do smartphone", "preco"),
    ("quanto vou pagar pelo notebook", "preco"),
    ("esse produto custa quanto", "preco"),

    # estoque
    ("tem notebook em estoque", "estoque"),
    ("o celular está disponível", "estoque"),
    ("vocês têm esse produto", "estoque"),
    ("tem o smartphone disponível", "estoque"),
    ("quero saber se o computador está em estoque", "estoque"),
    ("esse produto ainda está disponível", "estoque"),
    ("posso comprar esse produto agora", "estoque"),

    # pedido
    ("onde está meu pedido", "pedido"),
    ("quero acompanhar meu pedido", "pedido"),
    ("qual o status da minha entrega", "pedido"),
    ("meu pedido já foi enviado", "pedido"),
    ("quando meu pedido vai chegar", "pedido"),
    ("quero rastrear minha compra", "pedido"),
    ("como acompanho a entrega", "pedido"),
]

df = pd.DataFrame(dados, columns=["texto", "intencao"])
df



## Conhecendo os dados

Antes de treinar um modelo, observe os exemplos.

**Discussão:** essa abordagem baseada em regras pode funcionar para exemplos simples, mas tende a falhar quando a mesma intenção é expressa de várias maneiras.

É justamente aí que entra a classificação de intenções.


In [ ]:

print(df["intencao"].value_counts())



## Separando treinamento e teste

Vamos separar os exemplos em dois grupos:

- **treinamento:** exemplos utilizados pelo modelo para aprender;
- **teste:** exemplos reservados para verificar como o modelo se comporta diante de dados que não foram utilizados no treinamento.

Como o dataset é pequeno, esta divisão é apenas didática.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    df["texto"],
    df["intencao"],
    test_size=0.25,
    random_state=42,
    stratify=df["intencao"]
)

print("Treinamento:", len(X_train))
print("Teste:", len(X_test))



## Transformando texto em números

Computadores e algoritmos de ML não trabalham diretamente com o significado humano das frases. Precisamos transformar o texto em uma representação numérica.

Vamos utilizar:

**CountVectorizer → LogisticRegression**

O `CountVectorizer` transforma as palavras em características numéricas com base em sua ocorrência nos textos.

A `LogisticRegression` será utilizada como classificador das intenções.


In [ ]:

modelo = Pipeline([
    ("vetorizador", CountVectorizer()),
    ("classificador", LogisticRegression(max_iter=1000))
])

modelo.fit(X_train, y_train)

print("Modelo treinado!")



## Testando o classificador

Agora vamos fornecer mensagens que o modelo não recebeu exatamente dessa forma durante o treinamento.

Observe principalmente a **intenção prevista**.


In [ ]:

mensagens = [
    "quanto custa o smartphone",
    "vocês têm notebook disponível?",
    "quero acompanhar a minha entrega",
    "olá, preciso de ajuda",
]

previsoes = modelo.predict(mensagens)

for mensagem, previsao in zip(mensagens, previsoes):
    print(f"Mensagem: {mensagem}")
    print(f"Intenção: {previsao}")
    print("-" * 50)


### Teste de processo completo detalhado

In [ ]:
mensagem = "quanto custa o notebook?"


vetorizador = modelo.named_steps["vetorizador"]

print("# 1. Transformar texto em números: \n")
print( vetorizador.vocabulary_, '\n')

X = vetorizador.transform([mensagem])

print("# 2. Vetor de características: \n")
print(X.toarray(), "\n")

classificador = modelo.named_steps["classificador"]

print("# 3. Probabilidades: \n")
print(classificador.predict_proba(X)[0], "\n")

probabilidades = classificador.predict_proba(X)[0]

print("Mensagem:", mensagem, "\n")

for classe, probabilidade in zip(
    classificador.classes_,
    probabilidades
):
    print(f"{classe:10} → {probabilidade:.2%}")

print("\n Intenção prevista:", classificador.classes_[probabilidades.argmax()])

### O modelo aprende pesos associados às características. Esses pesos contribuem para decidir qual classe é mais provável.

In [ ]:
features = vetorizador.get_feature_names_out()

pesos_df = pd.DataFrame(
    classificador.coef_,
    columns=features,
    index=classificador.classes_
)

pesos_df


## Avaliando o modelo
Vamos observar as métricas de classificação.


In [ ]:

y_pred = modelo.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
y_train_pred = modelo.predict(X_train)
y_test_pred = modelo.predict(X_test)

print("Treinamento:")
print(accuracy_score(y_train, y_train_pred))

print("\nTeste:")
print(accuracy_score(y_test, y_test_pred))


### Como interpretar?

- **Precision:** entre as mensagens que o modelo classificou como uma determinada intenção, quantas estavam corretas?
- **Recall:** entre as mensagens que realmente pertenciam a uma intenção, quantas o modelo conseguiu encontrar?
- **F1-score:** combina precision e recall em uma única medida.

### Debate

Um modelo com 100% de acerto neste pequeno dataset significa que temos um chatbot pronto para produção?

**Não.**

O conjunto é pequeno e controlado. Em uma aplicação real, seria necessário avaliar generalização, qualidade das respostas, situações inesperadas, dados novos e outros aspectos.



## Pré-processamento
Para esta primeira implementação em português, vamos começar com uma normalização simples, evitando dependências adicionais de recursos linguísticos.


In [ ]:

def preprocessar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záàâãéêíóôõúç0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

exemplos = [
    "Olá! Quanto custa o Notebook?",
    "TEM CELULAR DISPONÍVEL???",
    "Quero acompanhar meu pedido #12345."
]

for texto in exemplos:
    print("Original:   ", texto)
    print("Processado: ", preprocessar(texto))
    print()



## Extração de entidades
Nesta atividade vamos implementar uma extração simples de número de pedido com expressão regular.

Isso é uma **regra**, não um modelo avançado de reconhecimento de entidades.


In [ ]:

def extrair_pedido(texto):
    resultado = re.search(r"(?:pedido|ordem|compra)\s*#?\s*(\d+)", texto.lower())
    return resultado.group(1) if resultado else None

for mensagem in [
    "Quero acompanhar o pedido 12345",
    "Qual o status da compra #9876?",
    "Quero rastrear meu pedido"
]:
    print(mensagem, "->", extrair_pedido(mensagem))



## Geração de linguagem natural (NLG)
O chatbot não vai gerar texto livre como um grande modelo de linguagem. Ele selecionará uma resposta adequada à intenção identificada.


In [ ]:

respostas = {
    "saudacao": [
        "Olá! Como posso ajudar?",
        "Oi! Em que posso ajudar você?",
        "Olá! Posso ajudar com preços, estoque ou pedidos."
    ],
    "preco": [
        "Posso ajudar a consultar o preço do produto.",
        "Claro! Vou verificar o preço para você."
    ],
    "estoque": [
        "Vou verificar a disponibilidade do produto.",
        "Claro! Vou consultar o estoque."
    ],
    "pedido": [
        "Vou ajudar você a acompanhar o pedido.",
        "Claro! Vamos verificar o status da sua entrega."
    ]
}

def gerar_resposta(intencao, mensagem):
    if intencao == "pedido":
        numero = extrair_pedido(mensagem)
        if numero:
            return f"Vou verificar o status do pedido #{numero}."

    return random.choice(respostas.get(
        intencao,
        ["Desculpe, não consegui entender sua solicitação."]
    ))



## Montando o chatbot

Agora vamos integrar:

**entrada → pré-processamento → NLU → intenção/entidade → NLG → resposta**


In [ ]:

def chatbot(mensagem):
    texto = preprocessar(mensagem)
    intencao = modelo.predict([texto])[0]
    resposta = gerar_resposta(intencao, mensagem)

    return {
        "mensagem": mensagem,
        "intencao": intencao,
        "resposta": resposta
    }

testes = [
    "Oi, tudo bem?",
    "Quanto custa o notebook?",
    "Tem celular disponível?",
    "Quero acompanhar o pedido 12345",
]

for mensagem in testes:
    resultado = chatbot(mensagem)
    print("Usuário:", resultado["mensagem"])
    print("Intenção:", resultado["intencao"])
    print("Chatbot:", resultado["resposta"])
    print("-" * 60)
